# 20.6 主动学习 / Active Learning

**中文**:上一节半监督**被动地**用无标注数据。**主动学习(Active Learning)** 换个角度:标注预算有限,与其随机挑数据让人标,不如让**模型主动挑出"最值得标"的样本**去请人标——把宝贵的标注预算花在刀刃上。核心信念:*"模型最没把握的样本,标了之后学到的最多。"* 一个好的主动学习策略,能**用少得多的标注达到同样的精度**,在标注昂贵(医疗、专家标注)时价值巨大。本节从零实现主动学习循环,并诚实地揭示:**策略选错反而不如随机。**
**English**: The previous section used unlabeled data **passively**. **Active learning** flips the view: with a limited labeling budget, rather than randomly picking data to label, let the **model actively choose the "most worth labeling" samples** to query — spending the precious labeling budget where it matters. The core belief: *"the samples the model is least sure about teach it the most once labeled."* A good active-learning strategy reaches the same accuracy with **far fewer labels**, hugely valuable when labeling is expensive (medical, expert annotation). This section implements the active-learning loop from scratch and honestly reveals: **a bad strategy can be worse than random.**

---

**中文**:**主动学习循环**:
**English**: The **active learning loop**:
1. 用当前(少量)标注数据训练模型。
   Train a model on the current (few) labels.
2. 用模型给**所有无标注数据打分**——哪些最"值得标"(最不确定)。
   Score all unlabeled data with the model — which are most "worth labeling" (most uncertain).
3. 挑出得分最高的一批,请**标注者(oracle)** 给它们打标签。
   Pick the top-scoring batch and ask the **oracle** to label them.
4. 加入训练集,回到第 1 步。
   Add them to the training set and go back to step 1.

**中文**:关键是**查询策略(query strategy)**——怎么衡量"不确定/值得标"。最常用的是**不确定性采样**:
**English**: The key is the **query strategy** — how to measure "uncertain / worth labeling." The most common is **uncertainty sampling**:
- **最小置信度(least confidence)**:选"最高类概率"最低的点(模型对预测最没底)。
  **Least confidence**: pick points with the lowest "top class probability" (the model is least sure).
- **边际采样(margin)**:选"前两名类概率之差"最小的点(模型在两类之间摇摆)。**通常最稳、最好用。**
  **Margin sampling**: pick points with the smallest "gap between the top-2 class probabilities" (the model wavers between two classes). **Usually the most robust.**
- **熵采样(entropy)**:选预测分布熵最大的点(在多个类间都不确定)。
  **Entropy**: pick points with the highest prediction entropy (uncertain across many classes).

**中文**:另一大类是**多样性/代表性**策略(选一批彼此不同、能覆盖数据分布的点),常与不确定性结合(纯不确定性可能一次选一堆相似的难样本,浪费预算)。
**English**: Another family is **diversity/representativeness** strategies (pick a batch of mutually different points covering the data distribution), often combined with uncertainty (pure uncertainty may pick a cluster of similar hard samples, wasting budget).

> 💡 **面试速查 / Interview cheat-sheet（★★ 少标注/降本必考）**
> **中文**:主动学习=**模型主动选最值得标的样本**去请人标, 少标注达同精度(标注贵时降本利器)。循环:训练→打不确定性分→选最不确定→标注→重训。**查询策略**:①**不确定性采样**(最小置信度/**边际(最稳)**/熵);②**委员会分歧(QBC)**;③**多样性/代表性**(防选一堆相似难样本);④期望模型改变。**关键坑(必考)**:①**采样偏差**——主动选的样本分布≠真实分布, 破坏 i.i.d., 且早期模型差时不确定性估计不可靠→纯不确定性有时**不如随机**(诚实!);②冷启动(初始模型太弱);③批量选要加多样性;④深度模型不确定性难估(用 MC-dropout/集成/BALD)。**vs 半监督**:AL 主动"选什么去标", SSL 被动"用无标注结构"——可结合。
> **English**: Active learning = **the model actively selects the most worth-labeling samples** to query, reaching the same accuracy with fewer labels (a cost-saver when labeling is expensive). Loop: train → score uncertainty → pick most uncertain → label → retrain. **Query strategies**: ① **uncertainty sampling** (least confidence / **margin (most robust)** / entropy); ② **query-by-committee (QBC, disagreement)**; ③ **diversity/representativeness** (avoid a cluster of similar hard samples); ④ expected model change. **Key pitfalls (interview favorites)**: ① **sampling bias** — actively selected samples' distribution ≠ true distribution, breaking i.i.d., and early weak models give unreliable uncertainty → pure uncertainty can be **worse than random** (honest!); ② cold start (initial model too weak); ③ batch selection needs diversity; ④ deep-model uncertainty is hard (use MC-dropout/ensembles/BALD). **vs semi-supervised**: AL actively "chooses what to label," SSL passively "uses unlabeled structure" — combinable.


In [ ]:

# ============================================================
# MNIST 主动学习:从少量标注开始, 每轮请人标一批 / MNIST active learning
# 中文:模拟"标注预算有限"。从30个标注起步, 每轮挑15个请oracle标注。对比"主动挑" vs "随机挑"。
# English: simulate a limited labeling budget. Start with 30 labels, each round query 15 for the oracle. Active vs random.
# ============================================================
import numpy as np, os, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from torchvision import datasets
from sklearn.linear_model import LogisticRegression
np.random.seed(0)
root=os.path.expanduser("~/.cache/dsfs_cv")
mn=datasets.MNIST(root, train=True, download=False)
X=(mn.data.float()/255.).view(-1,784).numpy(); y=mn.targets.numpy()
sub=np.random.permutation(len(X))[:8000]; X,y=X[sub],y[sub]
pool=np.arange(5000)                                         # 无标注池(oracle 知道真标签)/ unlabeled pool
test=np.arange(5000,8000)                                    # 测试集 / test set
print(f"无标注池 {len(pool)}, 测试集 {len(test)}; oracle 能给任意池样本标注")


**中文**:实现主动学习循环,对比四种策略:**随机**(基线)、**最小置信度**、**边际**、**熵**。看谁能用最少的标注达到最高精度。
**English**: Implement the active-learning loop comparing four strategies: **random** (baseline), **least confidence**, **margin**, **entropy**. See which reaches the highest accuracy with the fewest labels.


In [ ]:

# ============================================================
# 主动学习循环 + 四种查询策略 / active learning loop + four query strategies
# ============================================================
def run_active_learning(strategy, n_init=30, n_rounds=26, batch=15, seed=1):
    rng=np.random.default_rng(seed)
    labeled=list(rng.choice(pool, n_init, replace=False))    # 初始随机标注 / initial random labels
    curve=[]
    for r in range(n_rounds):
        clf=LogisticRegression(max_iter=300, C=0.1).fit(X[labeled], y[labeled])   # 训练 / train
        curve.append((len(labeled), clf.score(X[test], y[test])))                  # 记录精度 / record accuracy
        unl=np.array([i for i in pool if i not in set(labeled)])
        P=clf.predict_proba(X[unl])                          # 对无标注池打分 / score unlabeled
        if   strategy=="random":     score=rng.random(len(unl))
        elif strategy=="least_conf": score=1 - P.max(1)                            # 最高概率越低越不确定 / least confidence
        elif strategy=="margin":     s=np.sort(P,1); score=-(s[:,-1]-s[:,-2])      # 前两名差越小越不确定 / margin
        else:                        score=-(P*np.log(P+1e-12)).sum(1)             # 熵越大越不确定 / entropy
        pick=unl[np.argsort(score)[::-1][:batch]]            # 选最不确定的一批 / most uncertain batch
        labeled+=list(pick)
    return curve

curves={s:run_active_learning(s) for s in ["random","least_conf","margin","entropy"]}
print(f"{'策略/strategy':<16}{'@120标注':>10}{'@270标注':>10}{'@最终':>10}")
for s in ["random","least_conf","margin","entropy"]:
    c=curves[s]; a120=[a for n,a in c if n>=120][0]; a270=[a for n,a in c if n>=270][0]
    print(f"{s:<16}{a120:>10.3f}{a270:>10.3f}{c[-1][1]:>10.3f}")


**中文**:结果很有教育意义——**边际采样明显赢过随机**(同样标注量精度更高),但**熵和最小置信度反而可能不如随机**!下面画出学习曲线,并揭示这个诚实的反差。
**English**: The result is instructive — **margin sampling clearly beats random** (higher accuracy at the same budget), but **entropy and least-confidence can actually be worse than random**! Below we plot the learning curves and reveal this honest contrast.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 学习曲线 / learning curves
cols={"random":"#8C8C8C","least_conf":"#DD8452","margin":"#4C72B0","entropy":"#C44E52"}
names={"random":"随机 random","least_conf":"最小置信度","margin":"边际采样 margin","entropy":"熵采样 entropy"}
for s in ["random","margin","least_conf","entropy"]:
    ns=[n for n,a in curves[s]]; accs=[a for n,a in curves[s]]
    ax[0].plot(ns,accs,"o-",ms=3,color=cols[s],label=names[s],lw=2 if s=="margin" else 1)
ax[0].set_title("学习曲线:边际采样用更少标注达更高精度 / margin wins"); ax[0].set_xlabel("标注数量 #labels"); ax[0].set_ylabel("测试准确率"); ax[0].legend(fontsize=9)
# 标注:达到 0.82 各需多少标注 / labels needed to hit 0.82
target=0.82
for s in ["random","margin"]:
    hit=[n for n,a in curves[s] if a>=target]
    if hit: ax[0].axhline(target,ls=":",color="gray",alpha=0.5)
# ② 达到目标精度所需标注数 / labels needed for target accuracy
targets=[0.75,0.80,0.83]
width=0.35; xx=np.arange(len(targets))
for i,s in enumerate(["random","margin"]):
    needs=[next((n for n,a in curves[s] if a>=t), np.nan) for t in targets]
    ax[1].bar(xx+i*width,needs,width,label=names[s],color=cols[s])
ax[1].set_xticks(xx+width/2); ax[1].set_xticklabels([f"{t:.0%}" for t in targets])
ax[1].set_title("达到目标精度所需标注数(越少越好)/ labels to reach target"); ax[1].set_xlabel("目标准确率"); ax[1].set_ylabel("所需标注数"); ax[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/adv06_viz.png",dpi=80); plt.show()
# 定量:边际采样节省多少标注 / how many labels margin saves
for t in [0.80,0.83]:
    nr=next((n for n,a in curves["random"] if a>=t),None); nm=next((n for n,a in curves["margin"] if a>=t),None)
    if nr and nm: print(f"达到 {t:.0%}: 随机需 {nr} 标注, 边际采样只需 {nm} (省 {(1-nm/nr)*100:.0f}%)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **好的主动学习真能省标注**:边际采样在每个预算点都超过随机,达到同样精度所需的标注量明显更少。原理简单而深刻——**模型在两个类之间摇摆的样本,恰好落在决策边界附近,标了它最能帮模型把边界画准**;而随机采样会浪费预算在模型早已确信、标了也学不到新东西的"简单样本"上。在标注成本高昂的领域(医生标注影像、专家标注法律文书),这直接省真金白银。
2. **诚实的反差:策略选错反而不如随机**:熵采样和最小置信度在本例**没有稳定超过随机、甚至更差**。为什么？①**采样偏差**——主动选的样本不再是数据的随机代表,破坏了训练分布,可能过度采样某些"天生难分"的区域(如手写数字里模糊的 4/9);②**冷启动**——早期模型很弱,它给出的"不确定性"本身不可靠,基于它选样等于盲选甚至误选;③**缺多样性**——一次选一批最不确定的点,它们往往彼此相似(一堆长得像的难样本),信息冗余。**这戳破了"主动学习一定比随机好"的迷思——它是把双刃剑。**
3. **实战要点**:①**边际采样是最稳的默认起点**(比熵/最小置信度鲁棒);②批量选样一定要**加多样性**(如聚类后每簇选、或用 BADGE 等结合不确定性与多样性的方法);③深度模型的不确定性要用 **MC-dropout / 集成 / BALD** 靠谱地估;④**永远和随机基线比**(和 SSL 一样的诚实纪律);⑤别忘了主动学习和半监督**可以结合**(主动选少量关键点标注 + 无标注数据做一致性正则)。

**English**:
1. **Good active learning genuinely saves labels**: margin sampling beats random at every budget, needing far fewer labels for the same accuracy. The principle is simple yet deep — **samples where the model wavers between two classes lie right near the decision boundary, and labeling them most helps the model draw the boundary accurately**; random sampling wastes budget on "easy samples" the model is already confident about, which teach it nothing new. In high-labeling-cost domains (doctors labeling images, experts labeling legal documents), this saves real money.
2. **The honest contrast: a bad strategy can be worse than random**: entropy and least-confidence do **not reliably beat random here — even worse**. Why? ① **sampling bias** — actively chosen samples are no longer a random representation of the data, distorting the training distribution and possibly over-sampling inherently-hard regions (like ambiguous handwritten 4/9); ② **cold start** — early models are weak, so their "uncertainty" is itself unreliable, and selecting on it is near-blind or misguided; ③ **lack of diversity** — a batch of the most uncertain points is often mutually similar (a cluster of look-alike hard samples), with redundant information. **This punctures the myth that "active learning is always better than random" — it is a double-edged sword.**
3. **Practical keys**: ① **margin sampling is the most robust default** (more robust than entropy/least-confidence); ② batch selection must **add diversity** (e.g. cluster then pick per cluster, or use BADGE-style methods combining uncertainty and diversity); ③ estimate deep-model uncertainty reliably via **MC-dropout / ensembles / BALD**; ④ **always compare to a random baseline** (the same honest discipline as SSL); ⑤ remember active learning and semi-supervised **can be combined** (actively label a few key points + consistency regularization on unlabeled data).

> 💼 **实战视角 / Practical angle**
> **中文**:主动学习在**标注昂贵**的场景是降本利器:医疗影像标注、自动驾驶数据(选最难的场景标)、内容审核、法律/金融文档、工业质检。落地要点:①从**边际/BALD**起步, 批量必加多样性(BADGE/coreset);②**oracle 成本模型**——有时不同样本标注成本不同, 要按"信息/成本比"选;③**评估要诚实**——和随机基线比、用固定测试集、多个种子(主动学习方差大);④**流式主动学习**(数据一个个来, 决定标不标)vs 池式;⑤工具:`modAL`、`scikit-activeml`、`baal`(深度)。**特斯拉/自动驾驶**大量用主动学习挑"corner case"去人工标注。面试金句:*"主动学习让模型选最不确定的样本去标, 少标注达同精度; 边际采样最稳; 但采样偏差/冷启动/缺多样性会让纯不确定性有时不如随机——所以要加多样性、和随机基线对比。"*
> **English**: Active learning is a cost-saver where **labeling is expensive**: medical imaging, self-driving data (label the hardest scenes), content moderation, legal/financial documents, industrial inspection. Deployment keys: ① start with **margin/BALD**, always add diversity to batches (BADGE/coreset); ② model **oracle cost** — sometimes labeling costs differ per sample, so select by "information/cost ratio"; ③ **evaluate honestly** — compare to random, use a fixed test set and multiple seeds (active learning is high-variance); ④ **stream-based** active learning (data arrives one by one, decide to label or not) vs pool-based; ⑤ tools: `modAL`, `scikit-activeml`, `baal` (deep). **Tesla/self-driving** heavily use active learning to pick "corner cases" for human labeling. Interview line: *"Active learning lets the model choose the most uncertain samples to label, reaching the same accuracy with fewer labels; margin sampling is most robust; but sampling bias / cold start / lack of diversity can make pure uncertainty worse than random — so add diversity and always compare to a random baseline."*

---
### 小结 / Summary
- **中文**:主动学习让模型主动选最值得标的样本, 少标注达同精度(标注贵时降本); 循环:训练→打不确定性→选→标→重训。
- **English**: Active learning lets the model actively select the most worth-labeling samples, reaching the same accuracy with fewer labels; loop: train → score uncertainty → pick → label → retrain.
- **中文**:边际采样最稳且明显超随机; 但熵/最小置信度可能因采样偏差/冷启动/缺多样性反而不如随机。
- **English**: Margin sampling is most robust and clearly beats random; but entropy/least-confidence can be worse than random due to sampling bias / cold start / lack of diversity.
- **中文**:批量选样加多样性(BADGE), 深度不确定性用 MC-dropout/集成/BALD, 永远和随机基线比。
- **English**: Add diversity to batches (BADGE), estimate deep uncertainty via MC-dropout/ensembles/BALD, always compare to random.
